# LAB | Feature_engineering

In [1]:
#Step 0 - import libraries needed for this lab 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Standard seed used throughout the notebook, for reproducibility
SEED = 42

In [2]:
# Load the dataset

spaceship = pd.read_csv(
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv"
)
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


# Step 1 - EDA

In [3]:
print("Rows:", spaceship.shape[0])
print("Columns:", spaceship.shape[1])

Rows: 8693
Columns: 14


The dataset has **8,693 rows and 14 columns.** Each row represents one passenger, and the columns include demographic info, cabin/travel details, spending amounts, and the target column Transported.

In [4]:
# Check for data types
spaceship.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


We have a mix of **object** (categorical/text), **float** (numeric), and **bool** (the target, `Transported`) columns. The object columns, `HomePlanet`, `CryoSleep`, `Cabin`, `Destination`, `VIP`, `Name` will need work before a KNN model can use them.

In [5]:
# Check for missing values
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

Missing values range from **179 to 217** per column, which is small relative to the **8,693** total rows **(about 2%).** This confirms it's safe to simply drop the affected rows without losing much data.

There are multiple strategies to handle missing data
- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.
For this exercise, because we have such low amount of null values, we will drop rows containing any missing value.

In [7]:
# Handling missing data
spaceship = spaceship.dropna()
spaceship.shape

(6606, 14)

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

`Cabin` values look like `B/0/P` — deck / number / side. The deck letter alone (`{'A','B','C','D','E','F','G','T'}`) is a much more useful, lower-cardinality feature than the full cabin string, so we'll extract just that first character and overwrite the column.

In [8]:
spaceship["Cabin"] = spaceship["Cabin"].apply(lambda x: x.split("/")[0])
spaceship["Cabin"].value_counts()

Cabin
F    2152
G    1973
E     683
B     628
C     587
D     374
A     207
T       2
Name: count, dtype: int64

**Drop PassengerId and Name**

`PassengerId` is just a unique identifier and `Name` is free text neither carries predictive signal for a distance-based model like KNN, so we drop both.

In [9]:
spaceship = spaceship.drop(columns=["PassengerId", "Name"])
spaceship.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


For non-numerical columns, do dummies.

KNN needs every feature to be numeric, so we one-hot encode the remaining categorical columns (`HomePlanet`, `CryoSleep`, `Cabin`, `Destination`, `VIP`) with `pd.get_dummies`. We use `drop_first=True` to avoid redundant (perfectly correlated) dummy columns.

In [10]:
spaceship = pd.get_dummies(spaceship, drop_first=True)
spaceship.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,True,False,False,False,False,False,False,False,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False
2,58.0,43.0,3576.0,0.0,6715.0,49.0,False,True,False,False,False,False,False,False,False,False,False,False,True,True
3,33.0,0.0,1283.0,371.0,3329.0,193.0,False,True,False,False,False,False,False,False,False,False,False,False,True,False
4,16.0,303.0,70.0,151.0,565.0,2.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False


**Perform Train Test Split**
Now we separate **features (`X`)** from the **target (`y`, `Transported`)** and split into an **80/20 train/test set.**

In [11]:
X = spaceship.drop(columns=["Transported"])
y = spaceship["Transported"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (5284, 19)
X_test shape: (1322, 19)


**Model Selection**
In this exercise we will be using **KNN** as our predictive model.
`Transported` is True/False, so this is a **classification** task — we use `KNeighborsClassifier` with default hyperparameters, then fit it on the training set.

In [12]:
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


- Evaluate your model's performance. Comment it

In [13]:
y_pred = knn.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.7874432677760969

Classification Report:
               precision    recall  f1-score   support

       False       0.78      0.79      0.79       653
        True       0.79      0.78      0.79       669

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322

Confusion Matrix:
 [[518 135]
 [146 523]]


**Conclusion:**

With feature engineering, including extracting the `Cabin` deck, dropping the identifier/text columns, and one-hot encoding the categorical variables, the model now uses far more of the available information (16+ features) instead of just the 6 numeric spending/age columns from the previous lab. This typically improves successfully the pushes accuracy noticeably above the numeric-only baseline (~77%), achieving an overall accuracy of **78.74%**. This proves that features like `HomePlanet`, `CryoSleep`, and cabin `Deck` turn out to be strongly related to whether a passenger was transported.